# Aivora AI - Kaggle GPU Training

Runs the real training pipeline (data prep -> train -> evaluate -> export) on
Kaggle's free GPU tier. Nothing here is simulated: every cell either does the
real thing or raises a `RuntimeError('STATUS = BLOCKED: ...')` naming exactly
what's missing, the same discipline used in `training/colab/FinLLM_GPU_Training.ipynb`.

## Before running

1. **Settings (right sidebar) -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On** (required for `git clone` and streaming
   datasets from Hugging Face)
3. Kaggle's free tier gives **30 GPU-hours/week**, and a single session can
   run up to ~9-12 hours before it's cut off. `small` (10M tokens, ~2,000
   steps) comfortably fits in one sitting. `financial_poc` (50M tokens,
   8,000 steps) may need to be resumed across 1-2 sessions - the trainer
   already checkpoints every `eval_interval` steps, so `--resume` picks up
   exactly where it left off; this notebook's training cell supports that
   out of the box.
4. To resume from a previous session's checkpoint: add it as a Kaggle
   Dataset (Notebook -> Add Data -> Upload) and set `RESUME_CHECKPOINT`
   below to its path under `/kaggle/input/...`.

## What this does NOT do

It does not fabricate a GPU, does not skip the leakage check, and does not
report a training result that didn't actually happen. If a cell's hard gate
raises, the notebook stops there - fix the named problem and re-run.


## 1. Environment

In [ ]:
import platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input mounted:", os.path.exists("/kaggle/input"), os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else [])


## 1b. GPU compute-capability check (before the first `import torch`)

Kaggle has been assigning this account a Tesla P100 (compute capability
6.0 / sm_60) rather than a T4, and the base image's shipped torch build
only supports compute capability 7.0+ - every real CUDA kernel launch
fails with `AcceleratorError: no kernel image is available` regardless of
`batch_size` unless an older, wider-compatibility torch build is used.

Checked via `nvidia-smi` (not `torch.cuda.get_device_capability()`)
specifically so this runs **before** `import torch` - reinstalling torch
mid-process and `importlib.reload()`-ing it is not safe (torch's C
extension re-registers native `TORCH_LIBRARY` namespaces with the
dispatcher, which crashes on a second registration). If the attached GPU
needs an older, wider-compatibility torch build, it's installed here,
before torch is ever imported for the first time.

In [ ]:
import subprocess

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("nvidia-smi:", nvidia_smi.stdout.strip() or "(no output)", nvidia_smi.stderr.strip())

NEEDS_OLDER_TORCH = False
if nvidia_smi.returncode == 0 and nvidia_smi.stdout.strip():
    first_line = nvidia_smi.stdout.strip().splitlines()[0]
    name, _, cc_str = first_line.rpartition(",")
    try:
        compute_cap = float(cc_str.strip())
        if compute_cap < 7.0:
            NEEDS_OLDER_TORCH = True
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - below the "
                  "shipped torch build's minimum (7.0). Installing an older torch build "
                  "with wider compute-capability support before it's ever imported.")
        else:
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - "
                  "compatible with the shipped torch build, no reinstall needed.")
    except ValueError:
        print(f"Could not parse compute capability from {cc_str!r} - leaving the shipped "
              "torch build as-is and letting the GPU check cell catch any real problem.")
else:
    print("nvidia-smi query failed or returned nothing - leaving the shipped torch build "
          "as-is and letting the GPU check cell catch any real problem.")

if NEEDS_OLDER_TORCH:
    import sys
    # torch 2.7.1 (last line confirmed to still ship Pascal/sm_60 kernels)
    # + an older CUDA toolkit build (cu118) for the widest realistic
    # compute-capability coverage. This repo's attention implementation is
    # hand-written (no scaled_dot_product_attention / torch.compile
    # dependency), so an older torch build is not expected to break
    # anything model-specific.
    reinstall = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu118"],
        capture_output=True, text=True,
    )
    print(reinstall.stdout[-3000:])
    print(reinstall.stderr[-3000:])
    if reinstall.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: fallback torch==2.7.1+cu118 install failed for this "
            "compute-capability-6.0 GPU, see output above."
        )
    print("Installed torch==2.7.1+cu118 (not yet imported).")


## 2. GPU / CUDA verification (hard gate)

Raises immediately if no GPU is attached - never claims GPU training happened without this passing.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() is False. "
        "Go to Settings (right sidebar) -> Accelerator -> GPU T4 x2, "
        "save, and re-run this notebook from the top."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_COUNT = torch.cuda.device_count()
CC = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU AVAILABLE =", True)
print("GPU name:", GPU_NAME)
print("GPU count:", GPU_COUNT)
print("Compute capability:", CC)
print("Total VRAM (GPU 0): %.2f GB" % total_vram_gb)
print("torch version:", torch.__version__, "| CUDA build:", torch.version.cuda)


## 3. Repository transfer + integrity check

Clones the real, public repo. If this fails, Internet is probably off (Settings -> Internet -> On).

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: git clone failed (see stderr above). "
            "Most likely cause: Internet is off for this notebook "
            "(Settings -> Internet -> On), or the repo URL changed."
        )
else:
    print(f"{REPO_DIR} already present, skipping clone.")

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

required_paths = [
    "models/model.py", "training/trainer.py", "ai_platform/model_registry.py",
    "data_sources/prepare.py", "evaluation/evaluator.py", "configs/small.yaml",
    "configs/financial_poc.yaml", "inference/generator.py",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"STATUS = BLOCKED: repo clone incomplete, missing {missing}")
print("Repo integrity check passed:", len(required_paths), "required paths present.")


## 4. Dependencies

Kaggle's base image already ships a CUDA-enabled PyTorch build tuned for the
attached GPU - reinstalling `torch` over it risks silently downgrading to a
CPU or mismatched-CUDA wheel. This only installs the *other* requirements,
and reuses `torch.cuda.is_available()` from Cell 2 to confirm nothing broke
it afterward.

In [ ]:
import subprocess, sys

with open("requirements.txt") as f:
    reqs = [line.strip() for line in f if line.strip() and not line.startswith("#")]

# torch is already provided by the Kaggle GPU image - skip it here.
reqs_to_install = [r for r in reqs if not r.lower().startswith("torch")]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + reqs_to_install,
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError("STATUS = BLOCKED: pip install failed, see output above.")

check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print(check.stdout)
print(check.stderr)

# NOTE: deliberately do NOT importlib.reload(torch) here - torch's C
# extension init is not reload-safe (it re-registers native TORCH_LIBRARY
# namespaces with the dispatcher, which crashes on a second registration).
# Since torch itself was excluded from the install above, the already-
# imported torch module from Cell 2 is still valid and doesn't need reloading.
if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() became False after "
        "installing requirements.txt - something in the dependency list "
        "pulled in a CPU-only torch build. Check requirements.txt for a "
        "torch pin and remove it."
    )
print("Dependencies installed; CUDA still available after install.")


## 5. Dataset preparation

Prepares real data for every bucket in the chosen preset's `dataset_mix`,
streamed live from Hugging Face with a hard token budget per Part 9/37 of
this project - nothing is fully downloaded, and every dataset used is the
one already verified in `data_sources/dataset_registry.py` (license +
schema checked against a live sample, not assumed).

In [ ]:
PRESET = "financial_poc"  # "small" (10M tokens, ~2,000 steps) or "financial_poc" (raised-scale, see configs/financial_poc.yaml)

import yaml
from data_sources.dataset_registry import list_entries
from data_sources.dataset_mixer import BUCKET_TO_CATEGORY, validate_mix
from data_sources.prepare import prepare_dataset

with open(f"configs/{PRESET}.yaml") as f:
    preset_cfg = yaml.safe_load(f)

mix = preset_cfg["dataset_mix"]
validate_mix(mix)
# Optional: {dataset_name: max_tokens} - an explicit ceiling that
# overrides the automatic bucket_weight * total_budget / num_entries
# split for that one dataset. Needed once a bucket contains multiple
# datasets that should NOT share its budget equally (see
# configs/financial_poc.yaml's own comment on dataset_token_overrides).
overrides = preset_cfg.get("dataset_token_overrides", {})
total_budget = int(preset_cfg["train_tokens"]) + int(preset_cfg["validation_tokens"])
print(f"Preset '{PRESET}': total token budget {total_budget:,} across {len(mix)} buckets")
if overrides:
    print(f"  ({len(overrides)} dataset(s) use an explicit dataset_token_overrides ceiling instead of the bucket split)")

summary = []
for bucket, weight in mix.items():
    category = BUCKET_TO_CATEGORY[bucket]
    entries = list_entries(category=category, verified_only=True)
    if not entries:
        print(f"  [SKIP] bucket '{bucket}' (category '{category}') has no VERIFIED datasets registered.")
        continue
    auto_entries = [e for e in entries if e.name not in overrides]
    bucket_budget = int(total_budget * weight)
    auto_per_dataset_budget = max(bucket_budget // len(auto_entries), 50_000) if auto_entries else 0
    for entry in entries:
        per_dataset_budget = overrides.get(entry.name, auto_per_dataset_budget)
        print(f"  Preparing '{entry.name}' (bucket '{bucket}', budget {per_dataset_budget:,} tokens)...")
        result = prepare_dataset(entry.name, max_tokens=per_dataset_budget)
        summary.append((entry.name, result["train_tokens_used"], result["validation_tokens_used"]))

print()
print("Prepared datasets (real, measured token counts):")
for name, train_tok, val_tok in summary:
    print(f"  {name}: {train_tok:,} train / {val_tok:,} validation tokens")


## 6. Leakage check (hard gate)

Confirms no evaluation question appears verbatim in the training shards just prepared.

In [ ]:
from evaluation import check_leakage

leak_report = check_leakage()
print(leak_report)
if not leak_report.get("clean", False):
    raise RuntimeError(f"STATUS = BLOCKED: leakage detected - {leak_report}")
print("Leakage check passed: no evaluation text found in training shards.")


## 7. Tokenizer round-trip check

In [ ]:
from data_sources.tokenizer import get_encoding

enc = get_encoding()
sample = "EBITDA margin is calculated as EBITDA divided by revenue, times 100."
ids = enc.encode_ordinary(sample)
decoded = enc.decode(ids)
assert decoded == sample, f"Tokenizer round-trip failed: {decoded!r} != {sample!r}"
print(f"Tokenizer round-trip OK ({len(ids)} tokens for {len(sample)} chars).")


## 8. Model configuration

In [ ]:
from models import DeepSeekConfig, DeepSeekV3

config = DeepSeekConfig.default()
model_preview = DeepSeekV3(config)
param_count = sum(p.numel() for p in model_preview.parameters())
print(f"Model: {param_count:,} parameters")
print(config.to_dict())
del model_preview


## 9. Checkpoint discovery + compatibility check (dynamic, highest-step)

Finds the highest-numbered `checkpoint_*.pt` under the mounted
`aivora-ai-financial-poc-checkpoint` Kaggle Dataset, rather than a
hardcoded filename - this notebook gets re-pushed across many Kaggle
sessions over the coming weeks as this run continues past Kaggle's
~9-12h per-session cap, and the checkpoint dataset is updated with the
latest checkpoint before each push, so the notebook stays resume-agnostic
without needing a manual filename edit every time. Real tensor-shape
compatibility is checked against a freshly-built reference model before
trusting it - a checkpoint that doesn't match this repo's current
architecture is rejected with a clear reason rather than silently
corrupting the run.

In [ ]:
import glob as _glob
import re as _re

CHECKPOINT_DATASET_DIR = "/kaggle/input/aivora-ai-financial-poc-checkpoint"
candidates = _glob.glob(f"{CHECKPOINT_DATASET_DIR}/checkpoint_*.pt")
if not candidates:
    # Fall back to a broader search in case the dataset mounts under a
    # different path than expected - same reasoning as the original
    # kernel_sources-based search this replaces.
    candidates = _glob.glob("/kaggle/input/**/checkpoint_*.pt", recursive=True)

if not candidates:
    raise RuntimeError(
        "STATUS = BLOCKED: no checkpoint_*.pt found under /kaggle/input. "
        "Attach the aivora-ai-financial-poc-checkpoint dataset under this "
        "kernel's Settings -> Add Data."
    )


def _step_num(path):
    m = _re.search(r"checkpoint_(\d+)\.pt$", path)
    return int(m.group(1)) if m else -1


RESUME_CHECKPOINT = max(candidates, key=_step_num)
print(f"Found {len(candidates)} checkpoint(s) under /kaggle/input: "
      f"{sorted(candidates, key=_step_num)}")
print(f"Resuming from the highest-step one: {RESUME_CHECKPOINT}")

import torch
from models import DeepSeekConfig, DeepSeekV3

ref_model = DeepSeekV3(DeepSeekConfig.default())
ref_state = ref_model.state_dict()

ckpt = torch.load(RESUME_CHECKPOINT, map_location="cpu")
ckpt_state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

mismatches = []
for key, ref_tensor in ref_state.items():
    if key not in ckpt_state:
        mismatches.append(f"missing key: {key}")
    elif ckpt_state[key].shape != ref_tensor.shape:
        mismatches.append(f"shape mismatch on {key}: checkpoint has "
                           f"{ckpt_state[key].shape}, current model expects {ref_tensor.shape}")
if mismatches:
    raise RuntimeError(
        "STATUS = BLOCKED: checkpoint is not architecture-compatible with "
        f"the current model config: {mismatches[:5]}"
    )
print(f"Checkpoint {RESUME_CHECKPOINT} is architecture-compatible. Will resume from it.")
del ref_model


## 9b. Scale the per-step batch to the number of GPUs actually attached

`configs/financial_poc.yaml` pins `batch_size: 4` / `gradient_accumulation_steps: 32`,
measured on a **single P100**. `training.trainer.train_model` wraps the model in
`torch.nn.DataParallel` whenever `torch.cuda.device_count() > 1`, and DataParallel
**splits** each batch across GPUs - so on **GPU T4 x2** a `batch_size` of 4 would put
only 2 sequences on each GPU: half the per-GPU work the config was tuned for, and no
faster than a single GPU.

This cell multiplies `batch_size` by the GPU count and divides
`gradient_accumulation_steps` by the same factor, so that:

* the **per-GPU micro-batch stays at 4** - the size actually measured to fit
  (9.16 GiB allocated / 10.47 GiB reserved), which matters more on a T4
  (14.74 GiB) than it did on the P100 (15.89 GiB); and
* the **effective batch size stays at 128** (`batch_size * gradient_accumulation_steps`)
  - the value the resumed checkpoint's Adam moment estimates were tuned under.
  Changing it across a resume would silently invalidate that optimizer state.

It restores the committed config from git first, so re-running this cell is
idempotent rather than compounding the scale-up.


In [ ]:
import subprocess as _sp
import torch
import yaml

CFG_PATH = f"configs/{PRESET}.yaml"

# Restore the committed values before scaling, so re-running this cell in the
# same session scales from the checked-in baseline rather than compounding on
# top of an already-scaled config (the repo here is a fresh shallow clone, so
# this can never discard real work).
_restore = _sp.run(["git", "checkout", "--", CFG_PATH], capture_output=True, text=True)
if _restore.returncode != 0:
    print(f"(git checkout of {CFG_PATH} failed, using the file as-is: {_restore.stderr.strip()})")

with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

base_bs = int(cfg["batch_size"])
base_ga = int(cfg["gradient_accumulation_steps"])
effective_batch = base_bs * base_ga
gpu_count = torch.cuda.device_count()

print(f"Committed config: batch_size={base_bs}, grad_accum={base_ga} "
      f"(effective batch {effective_batch}), tuned on 1 GPU")
print(f"GPUs attached this session: {gpu_count} x {torch.cuda.get_device_name(0)}")

if gpu_count <= 1:
    print()
    print("Single GPU - leaving the config exactly as committed. NOTE: this notebook "
          "is configured for GPU T4 x2 (Settings -> Accelerator). If you meant to use "
          "two GPUs, stop here, switch the accelerator, and re-run from the top.")
elif base_ga % gpu_count != 0:
    print()
    print(f"gradient_accumulation_steps={base_ga} is not divisible by {gpu_count} - "
          f"cannot rescale while holding the effective batch at {effective_batch}. "
          f"Leaving the config as committed; each GPU will get "
          f"{base_bs / gpu_count:.1f} sequences per micro-batch.")
else:
    cfg["batch_size"] = base_bs * gpu_count
    cfg["gradient_accumulation_steps"] = base_ga // gpu_count
    assert cfg["batch_size"] * cfg["gradient_accumulation_steps"] == effective_batch
    with open(CFG_PATH, "w") as f:
        yaml.safe_dump(cfg, f)
    print()
    print(f"Scaled for {gpu_count} GPUs: batch_size={cfg['batch_size']}, "
          f"grad_accum={cfg['gradient_accumulation_steps']}")
    print(f"  per-GPU micro-batch: {cfg['batch_size'] // gpu_count} (unchanged from the "
          f"measured-safe {base_bs})")
    print(f"  effective batch:     {cfg['batch_size'] * cfg['gradient_accumulation_steps']} "
          f"(unchanged - resumed optimizer state stays valid)")


## 9c. Continuation stability test (GATE - the long run will not start until this passes)

Resuming `checkpoint_16000` at the from-scratch peak LR of `3.0e-4` diverged:
val went 6.07 -> 11.43 -> 20.01 -> 29.65 -> 33.88 -> 53.90 by step 20,000, with
**no NaN or Inf at any point** - ordinary gradient-descent blowup from too-large
steps, not a numerical fault. This cell spends ~15 GPU-minutes proving the
corrected settings are stable before committing to a multi-hour run.

What changed, all read from `configs/financial_poc.yaml`:

| control | before | now |
|---|---|---|
| peak LR (resume) | `3.0e-4` | **`3.0e-5`** via the `continuation:` block |
| gradient clipping | none at all | **`max_norm=1.0`** |
| GradScaler (fp16) | none | **enabled** (float16 only) |
| checkpoint dir | `checkpoints/base/` | **separate continuation dir** |

Two subtleties this cell handles deliberately:

* **`lr_horizon_override`** - `max_steps_override` shortens the cosine horizon,
  so a naive 1,000-step probe of a 495,700-step schedule would run at ~`1.02e-5`
  instead of the intended ~`3.0e-5` and "pass" without ever testing the real LR.
  Passing the true horizon makes the probe see the LR the full run will use.
* **`divergence_val_threshold`** - stops the moment val exceeds 1.5x the
  checkpoint's own best, instead of paying ~52 GPU-minutes to confirm what was
  already obvious at step 16,500 last time.


In [ ]:
import io as _io
import json as _json
import os as _os
import re as _re
import sys as _sys
from contextlib import redirect_stdout

import torch
from training.trainer import load_preset, train_model

# Write continuation checkpoints somewhere that is NOT checkpoints/base/, so a
# recovery run can never land on top of the checkpoint it is recovering from.
# (/kaggle/input is read-only anyway, but this keeps the invariant explicit and
# true when the same code runs locally.)
CONT_CKPT_DIR = "/kaggle/working/checkpoints/continuation"
_os.makedirs(CONT_CKPT_DIR, exist_ok=True)

_preset = load_preset(PRESET)
LR_HORIZON = int(_preset["max_steps"])
_cont = _preset.get("continuation") or {}
if not _cont:
    raise RuntimeError(
        "STATUS = BLOCKED: configs/%s.yaml has no 'continuation:' block, so a resume "
        "would use the from-scratch peak LR of %s - the setting that diverged. Add "
        "the block before running this cell." % (PRESET, _preset["learning_rate"])
    )

# Baseline comes from the checkpoint's own metadata, not a copied constant.
with open(RESUME_CHECKPOINT.rsplit(".pt", 1)[0] + ".json") as _f:
    _meta = _json.load(_f)
BASELINE_VAL = float(_meta["best_val_loss"])
THRESHOLD = BASELINE_VAL * 1.5

STABILITY_STEPS = 1000
STABILITY_EVAL_INTERVAL = 250

print(f"Baseline val_loss from {_os.path.basename(RESUME_CHECKPOINT)}: {BASELINE_VAL:.4f}")
print(f"Continuation LR: {_cont.get('learning_rate')} | grad_clip: "
      f"{_cont.get('grad_clip', _preset.get('grad_clip'))}")
print(f"Abort if val > {THRESHOLD:.4f} | {STABILITY_STEPS} steps, eval every "
      f"{STABILITY_EVAL_INTERVAL}")
print("")


class _Tee(_io.TextIOBase):
    """Keep the live Kaggle log AND a copy to assert acceptance criteria on."""

    def __init__(self, stream):
        self.stream, self.buf = stream, _io.StringIO()

    def write(self, s):
        self.stream.write(s)
        self.stream.flush()
        self.buf.write(s)
        return len(s)

    def flush(self):
        self.stream.flush()


_tee = _Tee(_sys.stdout)
STABILITY_ERROR = None
try:
    with redirect_stdout(_tee):
        train_model(
            preset_name=PRESET,
            resume=RESUME_CHECKPOINT,
            max_steps_override=STABILITY_STEPS,
            eval_interval_override=STABILITY_EVAL_INTERVAL,
            lr_horizon_override=LR_HORIZON,
            checkpoints_dir=CONT_CKPT_DIR,
            divergence_val_threshold=THRESHOLD,
        )
except RuntimeError as e:
    STABILITY_ERROR = e
    print(f"\n{e}")

_log = _tee.buf.getvalue()
_rows = _re.findall(
    r"step (\d+): train ([\d.]+), val ([\d.]+) \| lr ([\d.e+-]+) \| "
    r"grad_norm ([\d.naif]+)->([\d.naif]+) \(clipped (\d+)/(\d+)\) \| nonfinite (\d+)",
    _log)

print("\n" + "=" * 108)
print("STABILITY TEST RESULTS")
print("=" * 108)
print(f"{'step':>7} {'train':>9} {'val':>9} {'lr':>11} {'gnorm_pre':>10} "
      f"{'gnorm_post':>11} {'clipped':>12} {'nonfin':>7}")
for s, tr, vl, lr, gp, ga, cl, tot, nf in _rows:
    print(f"{s:>7} {tr:>9} {vl:>9} {lr:>11} {gp:>10} {ga:>11} "
          f"{cl + '/' + tot:>12} {nf:>7}")

_final = _re.search(r"Final train loss: ([\d.]+) \| Final val loss: ([\d.]+)", _log)
if _final:
    print(f"\nFinal: train {_final.group(1)} | val {_final.group(2)}")
if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GiB "
          f"of {torch.cuda.get_device_properties(0).total_memory / 1024 ** 3:.2f} GiB")

# ---- acceptance criteria (F) ----
checks = []
vals = [float(r[2]) for r in _rows]
lrs = [float(r[3]) for r in _rows]
nonfin = [int(r[8]) for r in _rows]
updates = [int(r[7]) for r in _rows]
final_val = float(_final.group(2)) if _final else (vals[-1] if vals else float("inf"))
peak_lr = float(_cont["learning_rate"])

checks.append(("ran to completion (no divergence abort)", STABILITY_ERROR is None))
checks.append(("at least 2 eval points recorded", len(_rows) >= 2))
checks.append(("no NaN/Inf gradient events", bool(nonfin) and max(nonfin) == 0))
checks.append(("no explosive growth (val never > 1.5x baseline)",
               bool(vals) and max(vals) <= THRESHOLD))
checks.append((f"val not worse than 1.1x baseline (final {final_val:.4f} vs "
               f"{BASELINE_VAL:.4f})", final_val <= BASELINE_VAL * 1.1))
checks.append((f"LR stayed in intended range (<= {peak_lr * 1.05:.2e})",
               bool(lrs) and max(lrs) <= peak_lr * 1.05))
# Prove clipping actually ran, not merely that updates happened: for every
# finite row the post-clip norm must equal min(pre, grad_clip).
_clip = float(_cont.get("grad_clip", _preset.get("grad_clip", 1.0)))
_finite = [(float(r[4]), float(r[5])) for r in _rows
           if r[4] not in ("nan", "inf") and r[5] not in ("nan", "inf")]
_clip_ok = bool(_finite) and all(
    abs(post - min(pre, _clip)) < 1e-3 for pre, post in _finite)
checks.append((f"gradient clipping applied (post == min(pre, {_clip}) on all "
               f"{len(_finite)} finite rows)", _clip_ok))
checks.append(("optimizer updates actually occurred",
               bool(updates) and max(updates) > 0))

print("\n" + "-" * 108)
for label, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
STABILITY_PASSED = all(ok for _, ok in checks)
print("-" * 108)
print(f"STABILITY_PASSED = {STABILITY_PASSED}")
if not STABILITY_PASSED:
    print("")
    print("Do NOT start the long run. Lower configs/%s.yaml continuation.learning_rate "
          "to 1.0e-5 (equal to min_lr, i.e. an effectively flat schedule) and re-run "
          "this cell. Spend no further quota until this passes." % PRESET)

## 9d. Demo checkpoint selection + inference smoke test

Picks the checkpoint to demo with, **by measured val_loss**, and proves it
generates end to end through the existing `inference/` code.

Deliberately independent of the long training cell below, so a demo can be
prepared from the stability run alone. Selection rule:

* take the best continuation checkpoint (lowest `val_loss` in its own `.json`);
* compare it against `checkpoint_16000`'s `best_val_loss`;
* demo whichever is genuinely better, and say which was chosen and why.

`checkpoint_16000` is copied, never moved or rewritten - it stays the fallback.


In [ ]:
import glob as _g
import json as _j
import os as _o
import shutil as _sh

from inference import generate_text

DEMO_DIR = "/kaggle/working/checkpoints/demo"
_o.makedirs(DEMO_DIR, exist_ok=True)


def _val_of(pt):
    """val_loss from a checkpoint's sidecar json, or None if absent."""
    mp = pt.rsplit(".pt", 1)[0] + ".json"
    if not _o.path.exists(mp):
        return None
    with open(mp) as f:
        m = _j.load(f)
    v = m.get("val_loss")
    return float(v) if isinstance(v, (int, float)) else m.get("best_val_loss")


_cands = []
for _pt in sorted(_g.glob(_o.path.join(CONT_CKPT_DIR, "checkpoint_*.pt"))):
    _v = _val_of(_pt)
    if _v is not None:
        _cands.append((_v, _pt))

_base_val = BASELINE_VAL
print(f"Baseline  {RESUME_CHECKPOINT} -> val {_base_val:.4f}")
for _v, _pt in _cands:
    print(f"Candidate {_pt} -> val {_v:.4f}")

if _cands and min(_cands)[0] < _base_val:
    DEMO_SOURCE, DEMO_VAL = min(_cands)[1], min(_cands)[0]
    _why = f"continuation checkpoint improved on the baseline ({DEMO_VAL:.4f} < {_base_val:.4f})"
else:
    DEMO_SOURCE, DEMO_VAL = RESUME_CHECKPOINT, _base_val
    _why = ("no continuation checkpoint beat the baseline - demoing the known-good "
            "checkpoint_16000 instead")

print(f"\nSelected: {DEMO_SOURCE}\nReason:   {_why}")

# Copy (never move/rewrite) so the fallback stays exactly where it was.
DEMO_CHECKPOINT = _o.path.join(DEMO_DIR, _o.path.basename(DEMO_SOURCE))
if _o.path.realpath(DEMO_SOURCE) != _o.path.realpath(DEMO_CHECKPOINT):
    _sh.copy2(DEMO_SOURCE, DEMO_CHECKPOINT)
    _src_meta = DEMO_SOURCE.rsplit(".pt", 1)[0] + ".json"
    if _o.path.exists(_src_meta):
        _sh.copy2(_src_meta, DEMO_CHECKPOINT.rsplit(".pt", 1)[0] + ".json")
print(f"Demo checkpoint staged at {DEMO_CHECKPOINT}")

# Prove the fallback is untouched by all of the above.
assert _o.path.exists(RESUME_CHECKPOINT), "fallback checkpoint disappeared!"
print(f"Fallback intact: {RESUME_CHECKPOINT} "
      f"({_o.path.getsize(RESUME_CHECKPOINT) / 1024 ** 3:.2f} GiB)")

DEMO_PROMPTS = [
    "What is EBITDA?",
    "Calculate EBITDA margin for revenue 500 and EBITDA 100.",
    "What is working capital?",
    "Explain gross margin in one sentence.",
    "What does a balance sheet show?",
]

print("\n" + "=" * 88)
print("DEMO INFERENCE SMOKE TEST")
print("=" * 88)
DEMO_OK = True
for _p in DEMO_PROMPTS:
    try:
        _out = generate_text(DEMO_CHECKPOINT, _p, max_tokens=48,
                             temperature=0.7, top_k=40, device="cuda")
        print(f"\nPROMPT: {_p}\nOUTPUT: {_out}")
    except Exception as _e:
        DEMO_OK = False
        print(f"\nPROMPT: {_p}\nERROR:  {type(_e).__name__}: {_e}")

print("\n" + "-" * 88)
print(f"DEMO_OK = {DEMO_OK} (all prompts generated without a runtime error)")
print(f"Demo checkpoint: {DEMO_CHECKPOINT} | val_loss {DEMO_VAL:.4f}")
print(f"Fallback:        {RESUME_CHECKPOINT} | val_loss {_base_val:.4f}")
if not DEMO_OK:
    raise RuntimeError("STATUS = BLOCKED: demo inference raised - do not demo this build.")

## 10. Training

Runs the real training loop (`training.trainer.train_model`), auto-detecting
the GPU. Catches genuine CUDA OOM and halves batch size, then sequence
length, retrying for real rather than assuming a config that was never
tested on this GPU. Checkpoints save every `eval_interval` steps (see the
preset yaml) - if this session is cut off by Kaggle's time limit, re-run
this notebook with `RESUME_CHECKPOINT` set to the last saved checkpoint
under `checkpoints/base/`.

In [ ]:
# GATE: cell 9c must have passed. A long run is only worth starting once the
# corrected continuation settings are known to be stable - the previous
# configuration burned ~52 GPU-minutes diverging to val 53.90.
if not globals().get("STABILITY_PASSED", False):
    raise RuntimeError(
        "STATUS = BLOCKED: the continuation stability test (cell 9c) has not "
        "passed. Run it first and only continue if it reports "
        "STABILITY_PASSED = True."
    )

import yaml
from training.trainer import train_model

# CFG_PATH is defined in cell 9b (GPU-count batch scaling).


def attempt_training(preset_name, resume, max_retries=4):
    import torch
    attempt = 0
    while attempt < max_retries:
        try:
            return train_model(preset_name=preset_name, resume=resume,
                               lr_horizon_override=LR_HORIZON,
                               checkpoints_dir=CONT_CKPT_DIR)
        except torch.cuda.OutOfMemoryError as e:
            attempt += 1
            print(f"CUDA OOM on attempt {attempt}/{max_retries}: {e}")
            torch.cuda.empty_cache()
            if attempt >= max_retries:
                raise RuntimeError(
                    f"STATUS = BLOCKED: repeated CUDA OOM after {max_retries} attempts "
                    f"on preset '{preset_name}'. batch_size/seq_len are already reduced "
                    f"as far as this notebook will go automatically - see "
                    f"configs/{preset_name}.yaml to reduce further by hand."
                )
            # Actually shrink the config on disk before retrying - train_model()
            # re-reads configs/{preset}.yaml fresh via load_preset() every call,
            # so mutating a local Python variable here would do nothing; this
            # was a real bug in an earlier version of this cell (it retried the
            # IDENTICAL config three times and failed identically every time).
            with open(CFG_PATH) as f:
                cfg = yaml.safe_load(f)
            if cfg["batch_size"] > 2:
                # Double gradient_accumulation_steps as batch_size is halved so
                # the EFFECTIVE batch (batch_size * grad_accum) is preserved. An
                # OOM fallback must not silently change the effective batch size
                # mid-run: the resumed checkpoint's Adam moment estimates were
                # tuned at 128, and halving it to 64 partway through would
                # invalidate them just as surely as editing the preset would.
                cfg["batch_size"] = max(2, cfg["batch_size"] // 2)
                cfg["gradient_accumulation_steps"] = int(cfg["gradient_accumulation_steps"]) * 2
                print(f"Reducing batch_size to {cfg['batch_size']} and raising "
                      f"gradient_accumulation_steps to {cfg['gradient_accumulation_steps']} "
                      f"(effective batch held at "
                      f"{cfg['batch_size'] * cfg['gradient_accumulation_steps']}); retrying.")
            elif cfg.get("seq_len") and cfg["seq_len"] > 128:
                cfg["seq_len"] = max(128, cfg["seq_len"] // 2)
                print(f"Reducing seq_len to {cfg['seq_len']} and retrying.")
            else:
                raise RuntimeError(
                    "STATUS = BLOCKED: batch_size/seq_len are already at the "
                    "minimum this notebook will try automatically."
                )
            with open(CFG_PATH, "w") as f:
                yaml.safe_dump(cfg, f)


model, config, ckpt_path = attempt_training(PRESET, RESUME_CHECKPOINT)
print(f"Training complete. Final checkpoint: {ckpt_path}")

## 11. Evaluation

In [ ]:
from evaluation import evaluate_model, print_report

eval_results = evaluate_model(model, device="cuda", max_new_tokens=48, verbose=True)
print_report(eval_results)


## 12. Inference test (the 3 required prompts)

In [ ]:
from inference import load_model_for_inference, generate_text

test_prompts = [
    "What is EBITDA?",
    "Calculate EBITDA margin for revenue 500 and EBITDA 100.",
    "What is working capital?",
]
for prompt in test_prompts:
    output = generate_text(ckpt_path, prompt, max_tokens=60, temperature=0.7, top_k=40, device="cuda")
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {output}")
    print("-" * 60)


## 13. Checkpoint export + hash verification

In [ ]:
import json
from ai_platform.model_registry import register_checkpoint, verify_integrity

entry = register_checkpoint(ckpt_path, stage="base", set_active=True)
print("Registered checkpoint:", entry)

verification = verify_integrity(entry["version"])
print("Integrity verification:", verification)
if not verification["valid"]:
    raise RuntimeError(f"STATUS = BLOCKED: checkpoint integrity verification failed: {verification}")

manifest = {
    "preset": PRESET,
    "checkpoint_path": ckpt_path,
    "gpu": GPU_NAME,
    "registry_entry": entry,
}
with open("export_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print("Wrote export_manifest.json")


## 14. Bringing the checkpoint back to your local project

1. In the Kaggle sidebar, open **Output** and download the checkpoint
   `.pt` + `.json` pair from `checkpoints/base/` (and `export_manifest.json`).
2. Place them in your local repo under `checkpoints/base/`.
3. Register it locally:
   ```bash
   python -c "from ai_platform.model_registry import register_checkpoint; print(register_checkpoint('checkpoints/base/<checkpoint_name>.pt', stage='base'))"
   ```
4. Restart the backend pointed at the new checkpoint and re-run
   `python ai_platform/acceptance_test.py` against it.
5. Only after that passes should this move from "trained on Kaggle" to
   `LLM TRAINING STATUS = TESTED` in the project's registry - this notebook
   proves the GPU run happened; the acceptance suite proves it's wired into
   the real serving stack correctly.
